In [1]:
import whisper
import librosa
import torch
import numpy as np
import demucs.separate
import soundfile as sf
from pathlib import Path

In [2]:

# Загружаем аудио
audio_file = "Баксанская.wav"
y, sr = librosa.load(audio_file, sr=16000)  # Whisper использует 16kHz
# 1. Разделяем вокал и инструментал с помощью demucs
print("Разделяем вокал и инструментал...")
demucs.separate.main(["--mp3", "--two-stems", "vocals", "-n", "mdx_extra", audio_file])

Разделяем вокал и инструментал...
Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in D:\MyPrograms\python\HackII\separated\mdx_extra
Separating track Баксанская.wav


100%|████████████████████████████████████████████████████████████████████████| 165.0/165.0 [01:28<00:00,  1.86seconds/s]
  0%|                                                                                  | 0.0/165.0 [00:02<?, ?seconds/s]

KeyboardInterrupt



In [ ]:
# Путь к инструментальной дорожке
instrumental_file = f"separated/mdx_extra/{Path(audio_file).stem}/vocals.mp3"
y_instrumental, sr_instrumental = librosa.load(instrumental_file, sr=16000)

# 2. Детекция фрагментов без вокала
print("Ищем фрагменты с музыкой (без вокала)...")
non_silent_intervals = librosa.effects.split(y_instrumental, top_db=30)  # Порог тишины

# Фильтруем фрагменты по длительности (20–30 секунд)
fragments = []
for start, end in non_silent_intervals:
    fragment_length = (end - start) / sr_instrumental
    if 20 <= fragment_length <= 30:
        fragments.append((start, end))
    elif fragment_length > 30:
        num_parts = int(fragment_length // 30)
        for j in range(num_parts):
            part_start = start + j * 30 * sr_instrumental
            part_end = part_start + 30 * sr_instrumental
            if (part_end - part_start) / sr_instrumental >= 20:
                fragments.append((part_start, part_end))
        remaining_start = start + num_parts * 30 * sr_instrumental
        if (end - remaining_start) / sr_instrumental >= 20:
            fragments.append((remaining_start, end))

# Печать найденных фрагментов
print("Найденные фрагменты (в секундах):")
for i, (start, end) in enumerate(fragments):
    start_sec = start / sr_instrumental
    end_sec = end / sr_instrumental
    print(f"Фрагмент {i+1}: {start_sec:.2f}–{end_sec:.2f} сек ({end_sec - start_sec:.2f} сек)")

# 3. Загружаем модель Whisper
model = whisper.load_model("large")

# 4. Обработка и проверка фрагментов
output_dir = Path("output_fragments")
output_dir.mkdir(exist_ok=True)

for i, (start, end) in enumerate(fragments):
    segment = y_instrumental[start:end]
    segment = whisper.pad_or_trim(segment)  # Приводим к длине 30 секунд

    # Создаем мел-спектрограмму с n_mels=128
    mel = whisper.log_mel_spectrogram(segment, n_mels=128).to(model.device)

    # Проверяем форму мел-спектрограммы
    print(f"Мел-спектрограмма (форма) для фрагмента {i+1}: {mel.shape}")

    # Распознавание текста
    options = whisper.DecodingOptions()
    result = whisper.decode(model, mel, options)

    # Сохранение фрагмента
    output_path = output_dir / f"fragment_{i+1}.wav"
    sf.write(output_path, segment, sr_instrumental)
    print(f"Сохранен: {output_path}")

    # Проверка на наличие текста
    if result.text.strip():
        print(f"Внимание: В фрагменте {i+1} обнаружен текст: {result.text}")
    else:
        print(f"Фрагмент {i+1}: Текста не обнаружено (чистая музыка).")

Ищем фрагменты с музыкой (без вокала)...
Найденные фрагменты (в секундах):
Фрагмент 1: 2.56–32.56 сек (30.00 сек)
Фрагмент 2: 32.56–62.56 сек (30.00 сек)
Фрагмент 3: 62.56–92.56 сек (30.00 сек)
Фрагмент 4: 92.56–122.56 сек (30.00 сек)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 5))
plt.plot(y_instrumental)
for start, end in fragments:
    plt.axvspan(start, end, color='green', alpha=0.3)
plt.title("Инструментальная дорожка с выделенными фрагментами")
plt.show()